# Vermont — Title 8 (Insurance chapters) → `data/vermont/ins_codes/*.md`

On **Justia**, **Title 8 — Banking and Insurance** is **[`/codes/vermont/title-8/`](https://law.justia.com/codes/vermont/title-8/)**. Banking and general provisions live in **lower-numbered chapters**; the **insurance** block is conventionally **Chapters 101–199** (e.g. Chapter 107 — Health Insurance).

This notebook downloads **chapters whose leading chapter number is between `CHAPTER_NUM_MIN` and `CHAPTER_NUM_MAX` (default 101–199)** (~**42** chapters, ~**1,009** sections). Adjust those bounds if you need a different slice.

**Cloudflare** often blocks plain **`httpx`**; this notebook uses **`curl_cffi`** with **`impersonate="chrome120"`**.

**Discovery:** One GET of the Title 8 index (**`div.primary-content`**), collect **`chapter-*`** links under **`/codes/vermont/title-8/`** in the numeric range; for each chapter page, collect **`section-*`** URLs under that chapter path.

**Download:** text from **`div.primary-content`**, with Justia boilerplate stripped. Files are **`VT_sec_<slug>.md`** where **`<slug>`** is the segment after **`section-`** (e.g. `4080` → `VT_sec_4080.md`; compound tails like `4051-d-1` are normalized). Display cites **`8 V.S.A. § …`**.

Config: **CHAPTER_NUM_MIN**, **CHAPTER_NUM_MAX**, **MAX_SECTIONS** (**0** = all), **MAX_DISCOVERY_PAGES** (**0** = no cap on chapter-page fetches). **REUSE_DISCOVERED_URLS** skips discovery when **`_vt_title8_ins_section_urls.txt`** exists.

Run with the **`ins_ipynb/`** directory as cwd. Then **`python -m app.ingest`** from the project root.


In [1]:
%pip install -q curl_cffi beautifulsoup4


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

BASE = "https://law.justia.com"
PATH_PREFIX = "/codes/vermont/title-8"
TITLE_INDEX = f"{BASE}{PATH_PREFIX}/"

OUT_DIR = Path("data") / "vermont" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CURL_IMPERSONATE = "chrome120"
REQUEST_DELAY_SEC = 0.12
TIMEOUT = 60.0

CHAPTER_NUM_MIN = 101
CHAPTER_NUM_MAX = 199

MAX_SECTIONS = 0
MAX_DISCOVERY_PAGES = 0

SKIP_EXISTING = True

DISCOVERED_LIST = OUT_DIR / "_vt_title8_ins_section_urls.txt"
REUSE_DISCOVERED_URLS = True

SKIP_PATH_SUBSTR = ("appendix", "chronological-history", "title-notes")


In [3]:
def curl_get(url: str) -> str:
    time.sleep(REQUEST_DELAY_SEC)
    r = curl_requests.get(url, impersonate=CURL_IMPERSONATE, timeout=TIMEOUT)
    r.raise_for_status()
    return r.text


def path_key(u: str) -> str:
    return urlparse(u).path.rstrip("/")


def skip_path(p: str) -> bool:
    low = p.lower()
    return any(s in low for s in SKIP_PATH_SUBSTR)


def chapter_leading_int(chapter_path: str) -> int | None:
    m = re.search(r"/chapter-([^/]+)/", chapter_path + "/", re.I)
    if not m:
        return None
    m2 = re.match(r"^(\d+)", m.group(1))
    return int(m2.group(1)) if m2 else None


def discover_chapter_urls() -> list[str]:
    html = curl_get(TITLE_INDEX)
    soup = BeautifulSoup(html, "html.parser")
    pc = soup.select_one("div.primary-content")
    if not pc:
        pc = soup
    chapters: set[str] = set()
    for a in pc.find_all("a", href=True):
        absu = urljoin(TITLE_INDEX, a["href"])
        pk = path_key(absu).lower()
        if not pk.startswith(PATH_PREFIX.lower() + "/chapter-"):
            continue
        if skip_path(pk):
            continue
        n = chapter_leading_int(pk)
        if n is None or n < CHAPTER_NUM_MIN or n > CHAPTER_NUM_MAX:
            continue
        chapters.add(absu if absu.endswith("/") else absu + "/")
    return sorted(
        chapters,
        key=lambda u: (chapter_leading_int(path_key(u)) or 0, path_key(u).lower()),
    )


def section_urls_on_chapter_page(chapter_url: str) -> list[str]:
    html = curl_get(chapter_url)
    soup = BeautifulSoup(html, "html.parser")
    pc = soup.select_one("div.primary-content")
    if not pc:
        pc = soup
    ch_pk = path_key(chapter_url).lower()
    found: set[str] = set()
    for a in pc.find_all("a", href=True):
        absu = urljoin(chapter_url, a["href"])
        pk = path_key(absu).lower()
        if "/section-" not in pk:
            continue
        if not pk.startswith(PATH_PREFIX.lower()):
            continue
        if skip_path(pk):
            continue
        if not pk.startswith(ch_pk + "/"):
            continue
        found.add(absu if absu.endswith("/") else absu + "/")
    return sorted(found, key=lambda u: label_sort_key(section_label_from_url(u)))


def discover_section_urls() -> list[str]:
    """Title 8 index → chapter pages in [CHAPTER_NUM_MIN, CHAPTER_NUM_MAX] → section URLs."""
    chapters = discover_chapter_urls()
    print(
        f"Chapters to scan: {len(chapters)} (leading chapter number {CHAPTER_NUM_MIN}–{CHAPTER_NUM_MAX})"
    )
    all_secs: list[str] = []
    seen: set[str] = set()
    fetches = 0
    for i, ch in enumerate(chapters, 1):
        if MAX_DISCOVERY_PAGES and fetches >= MAX_DISCOVERY_PAGES:
            print(f"Stopped discovery early (MAX_DISCOVERY_PAGES={MAX_DISCOVERY_PAGES})")
            break
        for su in section_urls_on_chapter_page(ch):
            if su not in seen:
                seen.add(su)
                all_secs.append(su)
        fetches += 1
        if i % 10 == 0:
            print(f"… discovery {i}/{len(chapters)} chapters, {len(all_secs)} sections so far")
    return sorted(all_secs, key=lambda u: label_sort_key(section_label_from_url(u)))


def section_label_from_url(url: str) -> str:
    path = path_key(url)
    low = path.lower()
    if "/section-" not in low:
        raise ValueError(f"not a section URL: {url!r}")
    return path.rsplit("/section-", 1)[1]


def label_sort_key(label: str) -> tuple:
    out: list[tuple[int, int | str]] = []
    for part in label.split("-"):
        if part.isdigit():
            out.append((0, int(part)))
        else:
            out.append((1, part.lower()))
    return tuple(out)


def label_to_display_citation(label: str) -> str:
    return f"8 V.S.A. § {label}"


def label_to_filename(label: str) -> str:
    safe = re.sub(r"[^0-9a-zA-Z]+", "_", label).strip("_").lower()
    return f"VT_sec_{safe}.md"


def extract_primary_text(html: str) -> tuple[str, str]:
    soup = BeautifulSoup(html, "html.parser")
    title_el = soup.find("title")
    title_txt = title_el.get_text(strip=True) if title_el else ""
    pc = soup.select_one("div.primary-content")
    if pc:
        text = pc.get_text("\n", strip=True)
    else:
        main = soup.find("main") or soup.find("article")
        text = main.get_text("\n", strip=True) if main else soup.get_text("\n", strip=True)
    return title_txt, text


def strip_justia_boilerplate(text: str) -> str:
    drop_prefixes = (
        "Go to Previous Versions",
        "View All Versions",
        "Learn more",
        "This media-neutral citation",
    )
    lines = text.split("\n")
    out: list[str] = []
    skip_until_substantive = True
    for line in lines:
        s = line.strip()
        if not s:
            if not skip_until_substantive:
                out.append("")
            continue
        if any(s.startswith(p) for p in drop_prefixes):
            continue
        if s.startswith("20") and ("Vermont Stat" in s or "V.S.A." in s):
            continue
        if s in {"Next", "Previous", "Universal Citation:"}:
            continue
        if s.startswith("Vermont Statutes Annotated"):
            continue
        skip_until_substantive = False
        out.append(s)
    return "\n".join(out).strip()


def download_vermont_title8_ins() -> dict[str, int]:
    if REUSE_DISCOVERED_URLS and DISCOVERED_LIST.exists() and DISCOVERED_LIST.stat().st_size > 50:
        raw = [ln.strip() for ln in DISCOVERED_LIST.read_text(encoding="utf-8").splitlines() if ln.strip()]
        all_urls = sorted(raw, key=lambda u: label_sort_key(section_label_from_url(u)))
        print(f"Loaded {len(all_urls)} section URLs from {DISCOVERED_LIST.name} (skipped discovery)")
    else:
        found = discover_section_urls()
        print(f"Discovered {len(found)} section URLs (Title 8, chapters {CHAPTER_NUM_MIN}–{CHAPTER_NUM_MAX})")
        all_urls = sorted(found, key=lambda u: label_sort_key(section_label_from_url(u)))
        DISCOVERED_LIST.write_text("\n".join(all_urls) + "\n", encoding="utf-8")

    todo = all_urls if not MAX_SECTIONS else all_urls[:MAX_SECTIONS]
    if MAX_SECTIONS:
        print(f"Limited downloads to first {len(todo)} sections (MAX_SECTIONS)")

    wrote = skipped = failed = 0
    for i, sec_url in enumerate(todo, 1):
        label = section_label_from_url(sec_url)
        disp = label_to_display_citation(label)
        dest = OUT_DIR / label_to_filename(label)
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
            skipped += 1
        else:
            try:
                html = curl_get(sec_url)
                head_t, body_t = extract_primary_text(html)
                body_t = strip_justia_boilerplate(body_t)
                title = head_t or f"Vermont Statutes {disp}"
                md = (
                    f"# {title}\n\n"
                    f"**Vermont Statutes Annotated — Title 8 (chapters {CHAPTER_NUM_MIN}–{CHAPTER_NUM_MAX}, insurance)**\n\n"
                    f"**Source (Justia mirror):** {sec_url}\n\n"
                    f"**Verify on official site:** [Vermont Legislature — Statutes](https://legislature.vermont.gov/statutes/)\n\n"
                    f"**Section (URL slug):** {label}\n\n"
                    f"**Citation (display):** {disp}\n\n"
                    f"---\n\n"
                    f"{body_t}\n"
                )
                dest.write_text(md, encoding="utf-8")
                wrote += 1
            except Exception as e:
                print(f"FAIL {label}: {e}")
                failed += 1
        if i % 200 == 0:
            print(f"… {i}/{len(todo)} (wrote={wrote} skipped={skipped} failed={failed})")

    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_vermont_title8_ins()


Chapters to scan: 42 (leading chapter number 101–199)
… discovery 10/42 chapters, 478 sections so far
… discovery 20/42 chapters, 614 sections so far
… discovery 30/42 chapters, 831 sections so far
… discovery 40/42 chapters, 978 sections so far
Discovered 1009 section URLs (Title 8, chapters 101–199)
… 200/1009 (wrote=200 skipped=0 failed=0)
… 400/1009 (wrote=400 skipped=0 failed=0)
… 600/1009 (wrote=600 skipped=0 failed=0)
… 800/1009 (wrote=800 skipped=0 failed=0)
… 1000/1009 (wrote=1000 skipped=0 failed=0)
Done. wrote=1009 skipped=0 failed=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/vermont/ins_codes


{'wrote': 1009, 'skipped': 0, 'failed': 0}

## Next step

`python -m app.ingest` from the project root.
